<h1>Polarize AI
<H1> <h2>"Sistema inteligente de análise política"<h2>

 INSTALAR BIBLIOTECAS



In [ ]:
!pip install -q google-api-python-client pandas openpyxl

IMPORTAR BIBLIOTECAS

In [ ]:
from googleapiclient.discovery import build
import pandas as pd
from google.colab import files
from google.colab import userdata

API KEY

In [ ]:
API_KEY = userdata.get('API_KEY')

CONECTAR NA API DO YOUTUBE

In [ ]:
youtube = build('youtube', 'v3', developerKey=API_KEY)

CONECTAR NA API DO YOUTUBE

In [ ]:
video_id = "T1v6g71X4_4"

PALAVRAS-CHAVE DOS CANDIDATOS

In [ ]:
candidatos = [
    'Renan Santos(missão)',
    'Flavio Bolsonaro(PL)',
    'Lula(PT)',
    'Ronaldo Caiado(PSD)',
    'Augusto Cury(Avante)'
]

In [ ]:
video_response = youtube.videos().list(
    part="snippet",
    id=video_id
).execute()

if not video_response.get("items"):
    raise ValueError("Vídeo não encontrado. Verifique o video_id.")

video = video_response["items"][0]

titulo = video["snippet"]["title"]
canal = video["snippet"]["channelTitle"]

# ============================================
# COLETA DOS COMENTÁRIOS
# ============================================

comentarios = []
next_page_token = None

while True:

    response = youtube.commentThreads().list(
        part="snippet",
        videoId=video_id,
        maxResults=100,
        pageToken=next_page_token,
        textFormat="plainText"
    ).execute()

    for item in response["items"]:

        snippet = item["snippet"]["topLevelComment"]["snippet"]

        comentario = snippet.get("textOriginal", "")
        likes = snippet.get("likeCount", 0)
        data = snippet.get("publishedAt", "")

        # Identificação simples do candidato
        texto = comentario.lower()

        candidato = "Não identificado"

        for nome in candidatos:
            if nome in texto:
                candidato = nome.capitalize()
                break

        comentarios.append({
            "video_id": video_id,
            "comentario": comentario,
            "likes": likes,
            "data": data,
            "video": titulo,
            "canal": canal,
            "candidato": candidato
        })

    next_page_token = response.get("nextPageToken")

    if not next_page_token:
        break

# ============================================
# DATAFRAME
# ============================================

df = pd.DataFrame(comentarios)

# Remover comentários vazios
df = df[df["comentario"].str.strip() != ""]

# Remover duplicados
df = df.drop_duplicates(subset=["comentario"])

# ============================================
# RESULTADO
# ============================================

print(f"Vídeo: {titulo}")
print(f"Canal: {canal}")
print(f"Comentários coletados: {len(df)}")

display(df.head())

# ============================================
# EXPORTAR CSV
# ============================================

import re

# Limpar o título para usar como nome de arquivo
titulo_arquivo = re.sub(r'[\\/*?:"<>|]', '', titulo)

# Substituir espaços por "_"
titulo_arquivo = re.sub(r'\s+', '_', titulo_arquivo)

# Remover caracteres especiais, mantendo letras, números e _
titulo_arquivo = re.sub(r'[^\w\-]', '', titulo_arquivo, flags=re.UNICODE)

# Criar nome final
nome_arquivo = f"comentarios_politicos_{titulo_arquivo}.csv"

df.to_csv(
    nome_arquivo,
    index=False,
    encoding="utf-8-sig"
)

# ============================================
# EXPORTAR XLSX
# ============================================

"""df.to_excel(
    "comentarios_politicos.xlsx",
    index=False
)"""

# ============================================
# DOWNLOAD
# ============================================

files.download(nome_arquivo)

Vídeo: Cury: Condenações do 8/1 foram exageradas e não houve golpe materializado | CNN Eleições
Canal: CNN Brasil
Comentários coletados: 789


,video_id,comentario,likes,data,video,canal,candidato
0,T1v6g71X4_4,Queria ver se a quebradeira fosse na casa dele...,0,2026-09-14T19:19:08Z,Cury: Condenações do 8/1 foram exageradas e nã...,CNN Brasil,Não identificado
1,T1v6g71X4_4,Punir pela intenção??? De onde tirou isso?,0,2026-09-14T18:36:31Z,Cury: Condenações do 8/1 foram exageradas e nã...,CNN Brasil,Não identificado
2,T1v6g71X4_4,O Cara é mais ensaboado que Muçum 🤷🏿‍♀️🤷🏿‍♀️🤷🏿...,1,2026-09-14T18:11:54Z,Cury: Condenações do 8/1 foram exageradas e nã...,CNN Brasil,Não identificado
3,T1v6g71X4_4,"Tava até pensando em votar em Cury, mais isso ...",0,2026-09-14T15:24:54Z,Cury: Condenações do 8/1 foram exageradas e nã...,CNN Brasil,Não identificado
4,T1v6g71X4_4,CONDENAÇÕES EXAGERADAS? \nTOTALMENTE INCONSTIT...,0,2026-09-14T15:20:45Z,Cury: Condenações do 8/1 foram exageradas e nã...,CNN Brasil,Não identificado


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>